<img src="https://www.bestdesigns.co/uploads/inspiration_images/4350/990__1511457498_404_walmart.png" alt="WALMART LOGO" />

# Walmart : predict weekly sales

## Company's Description 📇

Walmart Inc. is an American multinational retail corporation that operates a chain of hypermarkets, discount department stores, and grocery stores from the United States, headquartered in Bentonville, Arkansas. The company was founded by Sam Walton in 1962.

## Project 🚧

Walmart's marketing service has asked you to build a machine learning model able to estimate the weekly sales in their stores, with the best precision possible on the predictions made. Such a model would help them understand better how the sales are influenced by economic indicators, and might be used to plan future marketing campaigns.

## Goals 🎯

The project can be divided into three steps:

- Part 1 : make an EDA and all the necessary preprocessings to prepare data for machine learning
- Part 2 : train a **linear regression model** (baseline)
- Part 3 : avoid overfitting by training a **regularized regression model**

## Scope of this project 🖼️

For this project, you'll work with a dataset that contains information about weekly sales achieved by different Walmart stores, and other variables such as the unemployment rate or the fuel price, that might be useful for predicting the amount of sales. The dataset has been taken from a Kaggle competition, but we made some changes compared to the original data. Please make sure that you're using **our** custom dataset (available on JULIE). 🤓

## Deliverable 📬

To complete this project, your team should: 

- Create some visualizations
- Train at least one **linear regression model** on the dataset, that predicts the amount of weekly sales as a function of the other variables
- Assess the performances of the model by using a metric that is relevant for regression problems
- Interpret the coefficients of the model to identify what features are important for the prediction
- Train at least one model with **regularization (Lasso or Ridge)** to reduce overfitting


## Helpers 🦮

To help you achieve this project, here are a few tips that should help you: 

### Part 1 : EDA and data preprocessing

Start your project by exploring your dataset : create figures, compute some statistics etc...

Then, you'll have to make some preprocessing on the dataset. You can follow the guidelines from the *preprocessing template*. There will also be some specific transformations to be planned on this dataset, for example on the *Date* column that can't be included as it is in the model. Below are some hints that might help you 🤓

 #### Preprocessing to be planned with pandas

 **Drop lines where target values are missing :**
 - Here, the target variable (Y) corresponds to the column *Weekly_Sales*. One can see above that there are some missing values in this column.
 - We never use imputation techniques on the target : it might create some bias in the predictions !
 - Then, we will just drop the lines in the dataset for which the value in *Weekly_Sales* is missing.
 
**Create usable features from the *Date* column :**
The *Date* column cannot be included as it is in the model. Either you can drop this column, or you will create new columns that contain the following numeric features : 
- *year*
- *month*
- *day*
- *day of week*

**Drop lines containing invalid values or outliers :**
In this project, will be considered as outliers all the numeric features that don't fall within the range : $[\bar{X} - 3\sigma, \bar{X} + 3\sigma]$. This concerns the columns : *Temperature*, *Fuel_price*, *CPI* and *Unemployment*
 


**Target variable/target (Y) that we will try to predict, to separate from the others** : *Weekly_Sales*

 **------------**

 #### Preprocessings to be planned with scikit-learn

 **Explanatory variables (X)**
We need to identify which columns contain categorical variables and which columns contain numerical variables, as they will be treated differently.

 - Categorical variables : Store, Holiday_Flag
 - Numerical variables : Temperature, Fuel_Price, CPI, Unemployment, Year, Month, Day, DayOfWeek

### Part 2 : Baseline model (linear regression)
Once you've trained a first model, don't forget to assess its performances on the train and test sets. Are you satisfied with the results ?
Besides, it would be interesting to analyze the values of the model's coefficients to know what features are important for the prediction. To do so, the `.coef_` attribute of scikit-learn's LinearRegression class might be useful. Please refer to the following link for more information 😉 https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html

### Part 3 : Fight overfitting
In this last part, you'll have to train a **regularized linear regression model**. You'll find below some useful classes in scikit-learn's documentation :
- https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html#sklearn.linear_model.Ridge
- https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html#sklearn.linear_model.Lasso

**Bonus question**

In regularized regression models, there's a hyperparameter called *the regularization strength* that can be fine-tuned to get the best generalized predictions on a given dataset. This fine-tuning can be done thanks to scikit-learn's GridSearchCV class : https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

Also, you'll find here some examples of how to use GridSearchCV together with Ridge or Lasso models : https://alfurka.github.io/2018-11-18-grid-search/

---
# Solution
### Setup
If this cell installs something, restart the kernel and run all cells again.

Checks that the required libraries are installed and installs any that are missing. Its printed line is not saved in this notebook.

In [1]:
import sys, subprocess, importlib
REQUIRED = {'pandas': 'pandas', 'numpy': 'numpy', 'sklearn': 'scikit-learn', 'plotly': 'plotly', 'matplotlib': 'matplotlib'}
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('Installing:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
importlib.invalidate_caches()
print('Environment ready. Python:', sys.executable)

Imports the libraries and fixes the random seed so that the results can be reproduced. It hides one warning only: a cross-validation fold can miss a store with few rows.

In [2]:
import numpy as np, pandas as pd
import plotly.express as px, plotly.io as pio
from sklearn.model_selection import train_test_split, GridSearchCV, RepeatedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
pio.renderers.default = 'plotly_mimetype+notebook_connected'
SEED = 0
import warnings
warnings.filterwarnings('ignore', message='Found unknown categories')  # a CV fold can miss a store with few rows: it is then encoded as the reference store
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

## Part 1: EDA and preprocessing
### 1.1 Overview

This cell loads the Walmart file and shows its first rows, because we need to see the data before making any choice. The output shows 150 rows and 8 columns. Missing values are already visible in the first five rows.

In [3]:
df = pd.read_csv('data/Walmart_Store_sales.csv')
print(df.shape)
df.head()

(150, 8)


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,6.000,18-02-2011,"1,572,117.540",NaN,59.610,3.045,214.778,6.858
1,13.000,25-03-2011,"1,807,545.430",0.000,42.380,3.435,128.616,7.470
2,17.000,27-07-2012,NaN,0.000,NaN,NaN,130.720,5.936
3,11.000,NaN,"1,244,390.030",0.000,84.570,NaN,214.556,7.346
4,6.000,28-05-2010,"1,644,470.660",0.000,78.890,2.759,212.413,7.092


This cell counts the type, the missing values and the distinct values of each column, because missing values decide the cleaning steps. Only `Store` is complete, with 20 distinct stores. `Date` and `Temperature` miss 12% of their values, and the target `Weekly_Sales` misses 14 values, or 9.3%.

In [4]:
overview = pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum(), 'missing_pct': df.isna().mean() * 100,
                         'unique': df.nunique()})
overview

,dtype,missing,missing_pct,unique
Store,float64,0,0.000,20
Date,object,18,12.000,85
Weekly_Sales,float64,14,9.333,136
Holiday_Flag,float64,12,8.000,2
Temperature,float64,18,12.000,130
Fuel_Price,float64,14,9.333,120
CPI,float64,12,8.000,135
Unemployment,float64,15,10.000,104


This cell gives the basic statistics of each numeric column, to spot odd scales and extreme values. Weekly sales go from about 269,000 to 2.77 million USD, with a mean of 1.25 million. `Unemployment` reaches 14.3 while three quarters of its values are below 8.2. `Store` is a label, so its mean has no meaning.

In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Store,150.000,9.867,6.231,1.000,4.000,9.000,15.750,20.000
Weekly_Sales,136.000,"1,249,535.828","647,463.042","268,929.030","605,075.718","1,261,423.865","1,806,386.200","2,771,397.170"
Holiday_Flag,138.000,0.080,0.272,0.000,0.000,0.000,0.000,1.000
Temperature,132.000,61.398,18.379,18.790,45.587,62.985,76.345,91.650
Fuel_Price,136.000,3.321,0.478,2.514,2.852,3.451,3.706,4.193
CPI,138.000,179.899,40.275,126.112,131.971,197.909,214.935,226.969
Unemployment,135.000,7.598,1.577,5.143,6.598,7.470,8.150,14.313


150 weekly records from 20 stores, 8 columns, and missing values in every column except `Store` (8 to 12% of each column). Only 150 rows: every preprocessing choice that drops rows has a real cost, and scores computed on a small test set will be noisy.

### 1.2 Visualizations

This histogram shows the distribution of the target, because we need to know what we are trying to predict. The 136 known values are widely spread. The median (1.26 million) is close to the mean (1.25 million), so there is no long tail on one side. 42% of the weeks are below 1 million USD and 14% are above 2 million.

In [6]:
fig = px.histogram(df, x='Weekly_Sales', nbins=30, title='Distribution of weekly sales (USD)')
fig.show()

This chart compares the average weekly sales of each store, to check whether the store matters. It goes from about 302,500 USD for store 5 to 2.17 million USD for store 4, a factor of about 7. Each average rests on only 3 to 12 weeks, so these are rough levels.

In [7]:
store_sales = df.groupby('Store')['Weekly_Sales'].agg(['mean', 'count']).reset_index().sort_values('mean', ascending=False)
fig = px.bar(store_sales, x=store_sales['Store'].astype(int).astype(str), y='mean', hover_data=['count'],
             labels={'x': 'Store', 'mean': 'average weekly sales (USD)'}, title='Average weekly sales per store')
fig.show()

This cell gives the exact averages of the stores cited later: the lowest (store 5), the highest (store 4), store 1, which is the reference of the encoding, and store 12, which the outlier rule removes.

In [8]:
store_sales.set_index('Store').loc[[5, 4, 1, 12]].round(0)

,mean,count
Store,,
5.000,"302,500.000",8
4.000,"2,173,759.000",6
1.000,"1,562,508.000",9
12.000,"1,028,025.000",5


Sales levels differ by a factor of about 7 between stores. The dataset does not say why: it has no column for store size or location. The **store identity will be the main driver** of the prediction.

This box plot compares holiday weeks and normal weeks, because a holiday could change sales. Holiday weeks have a higher mean (1.33 million vs 1.24 million USD) and a higher median (1.64 million vs 1.25 million). But there are only 9 holiday weeks against 116 normal weeks, so this comparison is fragile.

In [9]:
fig = px.box(df.dropna(subset=['Holiday_Flag']), x=df.dropna(subset=['Holiday_Flag'])['Holiday_Flag'].map({0: 'normal week', 1: 'holiday week'}),
             y='Weekly_Sales', points='all', title='Weekly sales: holiday vs normal weeks', labels={'x': ''})
fig.show()

These four scatter plots show weekly sales against each economic or weather indicator, to look for a visible relation. Each point is one record. We read the strength of these relations in the next cell, with the correlations.

In [10]:
num_cols = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
long = df.melt(id_vars='Weekly_Sales', value_vars=num_cols).dropna()
fig = px.scatter(long, x='value', y='Weekly_Sales', facet_col='variable', facet_col_wrap=2,
                 title='Weekly sales vs economic and weather indicators', height=650)
fig.update_xaxes(matches=None, showticklabels=True)
fig.show()

This matrix gives the linear correlation between the numeric columns. It measures how two columns move together, from -1 to +1, and it does not prove a cause. With sales, the strongest value is CPI at -0.29, then temperature at -0.17. Unemployment (0.06), the holiday flag (0.04) and fuel price (-0.02) are close to zero. `Store` is not included, because it is a category.

In [11]:
corr = df[['Weekly_Sales', 'Holiday_Flag'] + num_cols].corr()
fig = px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r', zmin=-1, zmax=1, title='Correlation matrix (numeric columns)')
fig.show()

This chart places sales on a timeline, with one color per store, to look for a trend over time and for differences between stores. It keeps the 118 rows with both a date and a sales value. They go from 5 February 2010 to 19 October 2012.

In [12]:
dated = df.assign(Date=pd.to_datetime(df['Date'], format='%d-%m-%Y')).dropna(subset=['Date', 'Weekly_Sales']).sort_values('Date')
fig = px.scatter(dated, x='Date', y='Weekly_Sales', color=dated['Store'].astype(int).astype(str), title='Weekly sales over time, by store',
                 labels={'color': 'Store'})
fig.show()

No economic indicator has a strong linear relation with sales on its own (correlations are weak). `CPI` and `Unemployment` take values grouped by store (99% and 91% of their variance is between stores), so their effect is mixed with the store effect. Holiday weeks look higher, but there are few of them.

This cell measures the share of the variance of each indicator that lies between stores: the variance of the store averages divided by the total variance, on the rows where the indicator is known. It shows 99% for `CPI` and 91% for `Unemployment`, so they barely move inside a store, against 30% for `Temperature` and 18% for `Fuel_Price`.


In [13]:
def between_store_share(column):
    known = df[['Store', column]].dropna()
    return known.groupby('Store')[column].transform('mean').var() / known[column].var()


pd.Series({column: between_store_share(column) for column in num_cols}).round(3)


Temperature    0.303
Fuel_Price     0.175
CPI            0.993
Unemployment   0.908
dtype: float64

### 1.3 Preprocessing with pandas
**a. Drop rows with a missing target** (never impute the target).

This cell removes the rows with no sales value. We cannot learn from a row without a target, and filling it in would invent the answer. We go from 150 to 136 rows.

In [14]:
print('Rows before:', len(df))
df = df.dropna(subset=['Weekly_Sales'])
print('Rows after dropping missing Weekly_Sales:', len(df))

Rows before: 150
Rows after dropping missing Weekly_Sales: 136


**b. Features from `Date`**: year, month, day, day of week. Rows with a missing date keep NaN in these columns; they are imputed later in the scikit-learn pipeline (dropping them would cost 18 of the 136 rows with sales, as the output below shows).

This cell turns the date into year, month, day and day of week, because the model cannot use a date as text. The output checks the day of week: it is 4, meaning Friday, for all 118 known dates, and 18 rows have no date.

In [15]:
dates = pd.to_datetime(df['Date'], format='%d-%m-%Y')
df['Year'] = dates.dt.year
df['Month'] = dates.dt.month
df['Day'] = dates.dt.day
df['DayOfWeek'] = dates.dt.dayofweek
df = df.drop(columns='Date')
print(df['DayOfWeek'].value_counts(dropna=False))

DayOfWeek
4.000    118
NaN       18
Name: count, dtype: int64


`DayOfWeek` is always 4 (Friday): the dataset contains weekly sales recorded on the Friday of each week. A constant column carries no information, so it is **dropped**.

Removes `DayOfWeek`: it has the same value in every known row, so it cannot help the model tell one week from another.

In [16]:
df = df.drop(columns='DayOfWeek')

This cell shows the unemployment of store 12, before the outlier rule of the next cell removes it. Its values are regular, so they do not look like input errors.

In [17]:
df.loc[df['Store'] == 12, 'Unemployment'].agg(['count', 'min', 'max'])

count    5.000
min     13.503
max     14.313
Name: Unemployment, dtype: float64

**c. Outliers**: rows where `Temperature`, `Fuel_Price`, `CPI` or `Unemployment` is outside [mean - 3 std, mean + 3 std] are dropped. Missing values are kept at this stage (they are not outliers).

This cell applies the rule of the project brief: a row is removed when an indicator is more than 3 standard deviations from its mean. A missing value does not count as an outlier. Only `Unemployment` triggers the rule, above 12.52: 5 rows are removed, from 136 to 131. All 5 are store 12, whose unemployment is 13.5 to 14.3, so this store leaves the dataset and 19 stores remain. These bounds are computed before the train and test split, as the brief describes the step.

In [18]:
mask = pd.Series(True, index=df.index)
for c in num_cols:
    m, s = df[c].mean(), df[c].std()
    out = (df[c] < m - 3 * s) | (df[c] > m + 3 * s)
    print(f'{c:<13} bounds [{m - 3 * s:,.2f} ; {m + 3 * s:,.2f}]  outliers: {out.sum()}')
    mask &= ~out
df = df[mask]
print('Rows after outlier removal:', len(df))

Temperature   bounds [5.31 ; 116.40]  outliers: 0
Fuel_Price    bounds [1.88 ; 4.76]  outliers: 0
CPI           bounds [57.36 ; 298.82]  outliers: 0
Unemployment  bounds [2.81 ; 12.52]  outliers: 5
Rows after outlier removal: 131


### 1.4 Preprocessing with scikit-learn

This cell separates the target, keeps 20% of the rows for the final test, and defines the preparation of the features. We get 104 training rows and 27 test rows. The preparation is only declared here: it will learn its medians and scales from the training data, so the test set stays unseen.

In [19]:
target = 'Weekly_Sales'
cat_features = ['Store', 'Holiday_Flag']
num_features = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'Year', 'Month', 'Day']
X, y = df[cat_features + num_features], df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print('train:', X_train.shape, '| test:', X_test.shape)

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                      ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))]), cat_features),
])

train: (104, 9) | test: (27, 9)


- Numeric features: median imputation (robust to skew) + standardisation, so that coefficients are comparable (effect of +1 standard deviation).
- Categorical features: most frequent value imputation + one-hot encoding with `drop='first'` (avoids perfect collinearity in the linear regression). `Store` is a category, not a number: store 20 is not "20 times" store 1.
- Imputers and scaler are fitted on the train set only, inside the pipeline (no leakage).

## Part 2: Baseline, linear regression

This cell trains the baseline linear regression and scores it on both sets, to see how well it predicts and whether it overfits. R² is the share of the variance explained, where 1 is perfect and 0 is no better than predicting the mean. We get 0.974 on train and 0.936 on test. The mean absolute error on test is about 125,000 USD, or 11% of the average test week of 1.12 million USD.

In [20]:
def scores(model, name):
    p_tr, p_te = model.predict(X_train), model.predict(X_test)
    return {'model': name, 'R2 train': r2_score(y_train, p_tr), 'R2 test': r2_score(y_test, p_te),
            'RMSE train': root_mean_squared_error(y_train, p_tr), 'RMSE test': root_mean_squared_error(y_test, p_te),
            'MAE test': mean_absolute_error(y_test, p_te)}

linreg = Pipeline([('prep', preprocessor), ('model', LinearRegression())]).fit(X_train, y_train)
results = [scores(linreg, 'LinearRegression')]
pd.DataFrame(results).set_index('model')

,R2 train,R2 test,RMSE train,RMSE test,MAE test
model,,,,,
LinearRegression,0.974,0.936,"106,272.083","155,379.862","125,063.469"


Metrics: **R²** (share of the variance of weekly sales explained), **RMSE** and **MAE** in USD (typical error on a week's sales; MAE is less sensitive to a few large errors).

With a test set of 27 rows, one split can be lucky or unlucky. Cross-validation (5 folds repeated 10 times) on the training set gives a more reliable estimate and its spread.

This cell repeats the training on 5 folds, 10 times, using the training set only, because a single test of 27 rows can be lucky. It gives 50 validation scores. The mean validation R² is 0.939 with a standard deviation of 0.039, which confirms the test score. The gap with the train R² of 0.976 is about 0.04.

In [21]:
cv = RepeatedKFold(n_splits=5, n_repeats=10, random_state=SEED)

def cv_scores(model, name):
    r = cross_validate(model, X_train, y_train, cv=cv, scoring=('r2', 'neg_mean_absolute_error'), return_train_score=True)
    return {'model': name, 'CV R2 train': r['train_r2'].mean(), 'CV R2 val': r['test_r2'].mean(), 'CV R2 val std': r['test_r2'].std(),
            'CV MAE val': -r['test_neg_mean_absolute_error'].mean()}

cv_results = [cv_scores(Pipeline([('prep', preprocessor), ('model', LinearRegression())]), 'LinearRegression')]
pd.DataFrame(cv_results).set_index('model')

,CV R2 train,CV R2 val,CV R2 val std,CV MAE val
model,,,,
LinearRegression,0.976,0.939,0.039,"116,228.008"


### Coefficients

This cell extracts the model's coefficients, because the project asks which features matter. A store coefficient is the gap with store 1, the reference, with everything else unchanged. The largest are store 5 (-1.40 million USD), store 9 (-1.28 million) and store 3 (-1.26 million), three of the stores with the lowest sales.

In [22]:
def coefficients(pipe):
    names = pipe.named_steps['prep'].get_feature_names_out()
    names = [n.replace('num__', '').replace('cat__', '') for n in names]
    return pd.Series(pipe.named_steps['model'].coef_, index=names)

coefs = coefficients(linreg).sort_values()
fig = px.bar(coefs, orientation='h', title='LinearRegression coefficients (USD per week; numeric features: effect of +1 std)',
             labels={'value': 'coefficient (USD)', 'index': ''}, height=700)
fig.update_layout(showlegend=False); fig.show()
coefs.reindex(coefs.abs().sort_values(ascending=False).index).to_frame('coefficient').head(15)

,coefficient
Store_5.0,"-1,404,353.372"
Store_9.0,"-1,275,038.072"
Store_3.0,"-1,259,269.048"
Store_16.0,"-1,101,418.291"
Store_7.0,"-881,707.536"
Store_8.0,"-753,708.912"
Store_10.0,"730,670.651"
Store_14.0,"726,942.500"
Store_4.0,"661,398.547"
Store_17.0,"-627,061.613"


This table isolates the features other than the store, which the store coefficients would otherwise hide. Numeric features are standardised, so each value is the effect of +1 standard deviation. The largest are CPI (+100,900 USD) and unemployment (-97,600 USD). CPI has a negative correlation with sales (-0.29) but a positive coefficient here: its sign changes once the store is taken into account, which shows how fragile this value is. The holiday flag is negative (-47,400 USD), based on very few holiday weeks.

In [23]:
num_coefs = coefficients(linreg)[num_features + [c for c in coefficients(linreg).index if c.startswith('Holiday')]]
num_coefs.sort_values(key=abs, ascending=False).to_frame('coefficient (USD per week)')

,coefficient (USD per week)
CPI,"100,863.818"
Unemployment,"-97,631.959"
Month,"63,292.995"
Holiday_Flag_1.0,"-47,380.382"
Fuel_Price,"-40,814.742"
Temperature,"-37,954.320"
Day,"-33,575.683"
Year,"-19,332.336"


**Reading the coefficients**
- The `Store_x` dummies include by far the largest coefficients (from about 20,000 USD for the stores closest to store 1 to over a million USD): each one is the difference in weekly sales between store x and store 1 (the reference category dropped by the encoder), all else equal. **The store is the main driver of weekly sales.**
- Numeric features are standardised, so their coefficient is the change in weekly sales for +1 standard deviation of the feature. They are smaller than the largest store effects (see table above).
- `CPI` and `Unemployment` vary mostly *between* stores (mostly fixed per store), so part of their effect is already captured by the store dummies: their individual coefficients are unstable and should not be over-interpreted.

### Measuring the store effect
The coefficients suggest that the store dominates, but their size does not measure a share of explained variance. To measure it directly, we compare the cross-validated R² of the same linear regression with `Store` alone, then with every feature except `Store`.

This cell measures the weight of the store directly, because the size of a coefficient is not a share of explained variance. It runs the same regression three times with the same cross-validation. `Store` alone reaches 0.933. All the features except `Store` reach -0.123, which is worse than predicting the mean. All the features reach 0.939.

In [24]:
num_pipe, cat_pipe = preprocessor.transformers[0][1], preprocessor.transformers[1][1]
store_only = ColumnTransformer([('cat', cat_pipe, ['Store'])])
no_store = ColumnTransformer([('num', num_pipe, num_features), ('cat', cat_pipe, ['Holiday_Flag'])])

store_effect = pd.DataFrame([
    cv_scores(Pipeline([('prep', store_only), ('model', LinearRegression())]), 'Store only'),
    cv_scores(Pipeline([('prep', no_store), ('model', LinearRegression())]), 'All features except Store'),
    cv_scores(Pipeline([('prep', preprocessor), ('model', LinearRegression())]), 'All features'),
]).set_index('model')
store_effect

,CV R2 train,CV R2 val,CV R2 val std,CV MAE val
model,,,,
Store only,0.960,0.933,0.027,"113,948.538"
All features except Store,0.128,-0.123,0.188,"616,583.474"
All features,0.976,0.939,0.039,"116,228.008"


`Store` alone reaches a validation R² of 0.933. All the other features together reach -0.123: without the store, the model predicts worse than simply using the mean sales. Adding them to `Store` only moves R² from 0.933 to 0.939, a gain far smaller than the spread between folds (0.039). The store's identity explains almost all the variance the model captures.

This does not prove that the economy has no effect on sales. `CPI` and `Unemployment` are mostly fixed per store, so this dataset can hardly separate their effect from the store's.

## Part 3: Fight overfitting with regularization
The linear regression has 26 coefficients (18 store dummies, 1 holiday flag, 7 numeric features) for 104 training rows. Ridge (L2 penalty) shrinks all coefficients; Lasso (L1 penalty) can set some exactly to 0 (feature selection). The regularization strength `alpha` is tuned with **GridSearchCV** (bonus question), using the same repeated cross-validation.

This cell tries 31 values of the regularization strength `alpha` for Ridge and for Lasso, and keeps the best one by cross-validation, because `alpha` cannot be learned from the coefficients themselves. Ridge selects alpha = 0.0251 with a validation R² of 0.939. Lasso selects alpha = 681 with 0.941. These two alphas are on different scales, because the two penalties are not computed the same way.

In [25]:
ridge_search = GridSearchCV(Pipeline([('prep', preprocessor), ('model', Ridge())]),
                            {'model__alpha': np.logspace(-3, 3, 31)}, cv=cv, scoring='r2', return_train_score=True)
ridge_search.fit(X_train, y_train)
print('Ridge best alpha:', round(ridge_search.best_params_['model__alpha'], 4), '| CV R2:', round(ridge_search.best_score_, 4))

lasso_search = GridSearchCV(Pipeline([('prep', preprocessor), ('model', Lasso(max_iter=100_000))]),
                            {'model__alpha': np.logspace(0, 5, 31)}, cv=cv, scoring='r2', return_train_score=True)
lasso_search.fit(X_train, y_train)
print('Lasso best alpha:', round(lasso_search.best_params_['model__alpha'], 2), '| CV R2:', round(lasso_search.best_score_, 4))

Ridge best alpha: 0.0251 | CV R2: 0.9392


Lasso best alpha: 681.29 | CV R2: 0.9408


This chart shows the train and validation R² for each `alpha`, to see what the penalty does. With Ridge, the validation R² stays near 0.94 for small values, then falls to 0.86 at alpha = 1 and 0.38 at alpha = 10. With Lasso, it stays near 0.94 up to alpha = 681, its best value, is still 0.938 at 1,468, then falls to 0.64 near 15,000. The best scores are at the point where the penalty is still light, so the model did not need much correction.

In [26]:
curves = pd.concat([
    pd.DataFrame({'alpha': ridge_search.cv_results_['param_model__alpha'].astype(float), 'train': ridge_search.cv_results_['mean_train_score'],
                  'validation': ridge_search.cv_results_['mean_test_score'], 'model': 'Ridge'}),
    pd.DataFrame({'alpha': lasso_search.cv_results_['param_model__alpha'].astype(float), 'train': lasso_search.cv_results_['mean_train_score'],
                  'validation': lasso_search.cv_results_['mean_test_score'], 'model': 'Lasso'}),
]).melt(id_vars=['alpha', 'model'], var_name='set', value_name='R2')
fig = px.line(curves, x='alpha', y='R2', color='set', facet_col='model', log_x=True, markers=True,
              title='R2 vs regularization strength (repeated 5-fold CV)')
fig.update_xaxes(matches=None); fig.update_yaxes(range=[0, 1.02]); fig.show()

This cell gives the exact values behind the Ridge and Lasso curves, at the points cited in the comments: `alpha` = 1 and 10 for Ridge, and the value closest to 15,000 for Lasso.

In [27]:
ridge_curve = pd.Series(ridge_search.cv_results_['mean_test_score'], index=ridge_search.cv_results_['param_model__alpha'].astype(float))
lasso_curve = pd.Series(lasso_search.cv_results_['mean_test_score'], index=lasso_search.cv_results_['param_model__alpha'].astype(float))
print('Ridge validation R2:', ridge_curve.iloc[[(abs(ridge_curve.index - a)).argmin() for a in (1, 10)]].round(2).to_dict())
print('Lasso validation R2:', lasso_curve.iloc[[(abs(lasso_curve.index - 15000)).argmin()]].round(2).to_dict())

Ridge validation R2: {1.0: 0.86, 10.0: 0.38}
Lasso validation R2: {14677.992676220676: 0.64}


This cell scores the tuned models on the test set, next to the baseline. The test R² goes from 0.936 to 0.940 with Ridge and 0.948 with Lasso, and the mean absolute error goes from 125,000 to 115,000 USD with Lasso. These are 27 rows, so this small gain must be checked against cross-validation in the next cell.

In [28]:
ridge, lasso = ridge_search.best_estimator_, lasso_search.best_estimator_
results += [scores(ridge, f"Ridge (alpha={ridge_search.best_params_['model__alpha']:.3g})"),
            scores(lasso, f"Lasso (alpha={lasso_search.best_params_['model__alpha']:.3g})")]
cv_results += [cv_scores(ridge, 'Ridge'), cv_scores(lasso, 'Lasso')]
comparison = pd.DataFrame(results).set_index('model')
comparison

,R2 train,R2 test,RMSE train,RMSE test,MAE test
model,,,,,
LinearRegression,0.974,0.936,"106,272.083","155,379.862","125,063.469"
Ridge (alpha=0.0251),0.974,0.940,"106,488.809","150,893.249","122,447.610"
Lasso (alpha=681),0.973,0.948,"108,184.535","139,922.296","115,307.214"


This cell compares the models in cross-validation and computes the overfitting gap, the difference between train and validation R². The gap is 0.038 for the baseline, 0.037 for Ridge and 0.034 for Lasso. The validation R² goes from 0.939 to 0.941, far less than its standard deviation of about 0.04. There was little overfitting to correct.

In [29]:
cv_comparison = pd.DataFrame(cv_results).set_index('model')
cv_comparison['overfitting gap (train - val R2)'] = cv_comparison['CV R2 train'] - cv_comparison['CV R2 val']
cv_comparison

,CV R2 train,CV R2 val,CV R2 val std,CV MAE val,overfitting gap (train - val R2)
model,,,,,
LinearRegression,0.976,0.939,0.039,"116,228.008",0.038
Ridge,0.976,0.939,0.038,"114,768.291",0.037
Lasso,0.975,0.941,0.036,"113,449.439",0.034


This cell compares the coefficients of the three models, to see what the penalty changed. Lasso sets only one coefficient to 0: store 6, which is close to the reference store. CPI goes from 100,900 to 68,000 USD, and unemployment from -97,600 to -72,800 USD. Those are the two features that overlap with the store, but they are not the only ones reduced.

In [30]:
coef_compare = pd.DataFrame({'LinearRegression': coefficients(linreg), 'Ridge': coefficients(ridge), 'Lasso': coefficients(lasso)})
zeroed = coef_compare.index[coef_compare['Lasso'].abs() < 1e-6].tolist()
print('Coefficients set to 0 by Lasso:', zeroed if zeroed else 'none')
fig = px.bar(coef_compare.loc[num_features + ['Holiday_Flag_1.0']].reset_index().melt(id_vars='index'), x='value', y='index', color='variable',
             barmode='group', orientation='h', title='Coefficients of the non-store features, by model', labels={'index': '', 'value': 'USD per week'})
fig.show()
coef_compare.round(0)

Coefficients set to 0 by Lasso: ['Store_6.0']


,LinearRegression,Ridge,Lasso
Temperature,"-37,954.000","-37,524.000","-35,895.000"
Fuel_Price,"-40,815.000","-39,408.000","-35,598.000"
CPI,"100,864.000","95,691.000","67,990.000"
Unemployment,"-97,632.000","-88,465.000","-72,795.000"
Year,"-19,332.000","-16,677.000","-13,043.000"
Month,"63,293.000","62,157.000","58,936.000"
Day,"-33,576.000","-32,945.000","-30,592.000"
Store_2.0,"398,263.000","412,670.000","399,546.000"
Store_3.0,"-1,259,269.000","-1,233,422.000","-1,219,935.000"
Store_4.0,"661,399.000","677,316.000","627,494.000"


This chart compares predicted and actual sales on the test set, for the model with the best test R², here Lasso. The closer a point is to the dashed line, the better the prediction. The test score is only used to choose which chart to show here, not to select the final model. 44% of the 27 predictions are within 10% of the actual value, and the largest error is about 307,000 USD.

In [31]:
best_name = comparison['R2 test'].idxmax()
best = {'LinearRegression': linreg}.get(best_name, ridge if best_name.startswith('Ridge') else lasso)
pred = best.predict(X_test)
fig = px.scatter(x=y_test, y=pred, labels={'x': 'actual weekly sales', 'y': 'predicted weekly sales'},
                 title=f'{best_name}: predicted vs actual (test set)')
lim = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
fig.add_shape(type='line', x0=lim[0], y0=lim[0], x1=lim[1], y1=lim[1], line=dict(dash='dash'))
fig.show()

## Conclusion

This cell gathers the final scores of the three models, on the test set and in cross-validation, as the basis for the conclusion. The three models are very close. The average test week is 1.12 million USD, which gives a scale for the errors.

In [32]:
c = comparison; v = cv_comparison
for name in c.index:
    print(f"{name:<28} test R2 {c.loc[name, 'R2 test']:.3f} | test MAE {c.loc[name, 'MAE test']:,.0f} USD | train R2 {c.loc[name, 'R2 train']:.3f}")
print()
for name in v.index:
    print(f"{name:<18} CV validation R2 {v.loc[name, 'CV R2 val']:.3f} (std {v.loc[name, 'CV R2 val std']:.3f}), train-val gap {v.loc[name, 'overfitting gap (train - val R2)']:.3f}")
print(f"\nMean weekly sales in the test set: {y_test.mean():,.0f} USD")

LinearRegression             test R2 0.936 | test MAE 125,063 USD | train R2 0.974
Ridge (alpha=0.0251)         test R2 0.940 | test MAE 122,448 USD | train R2 0.974
Lasso (alpha=681)            test R2 0.948 | test MAE 115,307 USD | train R2 0.973

LinearRegression   CV validation R2 0.939 (std 0.039), train-val gap 0.038
Ridge              CV validation R2 0.939 (std 0.038), train-val gap 0.037
Lasso              CV validation R2 0.941 (std 0.036), train-val gap 0.034

Mean weekly sales in the test set: 1,116,180 USD


- **Performance:** the baseline linear regression explains about 94% of the variance of weekly sales on the test set (R² 0.936, MAE about 125k USD for weekly sales averaging 1.1 M USD, so roughly 11% error). Cross-validation confirms it (validation R² 0.939, std 0.04).
- **Important features:** the store's identity explains almost all the variance the model captures: in cross-validation, `Store` alone reaches R² 0.933, all the other features together reach -0.123, and all features reach 0.939. The store dummies are also the largest coefficients (up to 1.4 M USD per week between stores). Among other features, CPI (+) and unemployment (-) have the largest coefficients (about 100k USD per standard deviation), then month, fuel price and temperature (about 38k to 63k USD). CPI and unemployment are mostly fixed per store, so they overlap with the store effect and their individual values are fragile. The holiday flag comes out slightly negative, against intuition: there are very few holiday weeks left after cleaning, too few for a reliable estimate.
- **Overfitting:** limited for the baseline (train-validation R² gap of about 0.04). Ridge and Lasso tuned with GridSearchCV reduce the gap slightly (Lasso 0.034) and improve the test MAE (Lasso: 115k USD vs 125k USD), but the cross-validated gain (R² 0.941 vs 0.939) is well within one standard deviation: on this dataset regularization helps a little, it does not change the picture. Lasso set only one coefficient to 0 (store 6, which is close to the reference store 1).
- **Limits:** 150 rows for 20 stores, then 131 rows for 19 stores after cleaning (the outlier rule removes every row of store 12), so a few weeks per store; conclusions on economic indicators are fragile. With the full Kaggle history (6,435 rows), the same pipeline would give much more reliable coefficients. This model cannot plan marketing campaigns: the file contains no campaign data.